# Scanpy Tutorial: 3k PBMC Preprocessing and Clustering

This notebook is a compact tutorial for the standard Scanpy single-cell RNA-seq workflow using the 10x Genomics 3k peripheral blood mononuclear cell (PBMC) dataset. PBMC is not thymus data, but it is small, public, and useful for testing the same syntax that will later be applied to thymic stromal populations in the ThymusLOXScan project.

The goal is to practice the full analysis pattern: load data, perform quality control, normalize expression, select informative genes, reduce dimensionality, build a neighborhood graph, compute UMAP, cluster cells, identify marker genes, and query genes of interest.

## 1. Setup

Before starting the analysis, we import the packages used throughout the workflow and configure Scanpy plotting. Keeping imports in one place makes the notebook easier to rerun and easier to adapt for thymus data later.

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

sc.settings.verbosity = 3
sc.settings.set_figure_params(dpi=100, facecolor="white")

%matplotlib inline

## 2. Download and Load the PBMC 3k Dataset

We use `scanpy.datasets.pbmc3k()` because it downloads a known public 10x Genomics dataset and returns it as an AnnData object. AnnData is the central container used by Scanpy: expression values live in `adata.X`, cell-level metadata live in `adata.obs`, and gene-level metadata live in `adata.var`.

Using this dataset first gives us a reproducible test case before moving to larger thymus scRNA-seq data.

In [ ]:
adata = sc.datasets.pbmc3k()
adata.var_names_make_unique()

adata

## 3. Quality Control Metrics

Quality control removes cells that are likely empty droplets, damaged cells, or technical outliers. Three common metrics are especially useful:

- `n_genes_by_counts`: the number of genes detected in each cell. Very low values can indicate empty droplets or poor capture; very high values can indicate doublets.
- `total_counts`: the total number of transcript counts per cell. Extreme values can reflect low-quality cells or multiplets.
- `pct_counts_mt`: the percentage of counts from mitochondrial genes. High mitochondrial signal often indicates stressed or dying cells.

These thresholds are deliberately simple for a tutorial. In a real thymus analysis, thresholds should be inspected per sample because tissue dissociation, age, and cell type composition can shift QC distributions.

In [ ]:
adata.var["mt"] = adata.var_names.str.startswith("MT-")
sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], percent_top=None, log1p=False, inplace=True)

sc.pl.violin(
    adata,
    ["n_genes_by_counts", "total_counts", "pct_counts_mt"],
    jitter=0.4,
    multi_panel=True,
)

sc.pl.scatter(adata, x="total_counts", y="pct_counts_mt")
sc.pl.scatter(adata, x="total_counts", y="n_genes_by_counts")

## 4. Filter Low-Quality Cells and Lowly Detected Genes

Filtering improves downstream analysis by removing cells and genes that mostly contribute noise. Here we keep cells with a reasonable number of detected genes, exclude high-count outliers, and remove cells with elevated mitochondrial percentage.

We also remove genes detected in fewer than three cells. Genes observed in only one or two cells usually do not help clustering and can slow down the analysis.

In [ ]:
adata = adata[
    (adata.obs["n_genes_by_counts"] >= 200)
    & (adata.obs["n_genes_by_counts"] <= 2500)
    & (adata.obs["total_counts"] <= 20000)
    & (adata.obs["pct_counts_mt"] < 5),
].copy()

sc.pp.filter_genes(adata, min_cells=3)

adata

## 5. Normalize Counts and Log Transform

Cells differ in sequencing depth, so raw counts are not directly comparable across cells. `normalize_total(target_sum=10000)` rescales each cell to the same total count depth, making expression values more comparable.

The `log1p` transformation then compresses large expression values and makes expression distributions easier to model and visualize. We keep a copy of the normalized data in `adata.raw` so marker-gene plots can refer back to this expression scale after later transformations.

In [ ]:
sc.pp.normalize_total(adata, target_sum=10000)
sc.pp.log1p(adata)

adata.raw = adata

## 6. Select Highly Variable Genes

Single-cell datasets contain thousands of genes, but many genes are uninformative for distinguishing cell states. Highly variable genes capture the strongest biological differences between cells after accounting for average expression.

Selecting the top 2,000 highly variable genes focuses PCA and clustering on informative signal rather than housekeeping genes or sparse background noise.

In [ ]:
sc.pp.highly_variable_genes(adata, n_top_genes=2000)

sc.pl.highly_variable_genes(adata)

adata = adata[:, adata.var["highly_variable"]].copy()

## 7. Scale the Data

Scaling centers each gene to mean zero and unit variance. This prevents highly expressed genes from dominating PCA purely because of their magnitude.

The `max_value=10` argument clips extreme values, which reduces the influence of rare very-high-expression outliers on dimensionality reduction.

In [ ]:
sc.pp.scale(adata, max_value=10)

## 8. Principal Component Analysis

PCA compresses the highly variable gene expression matrix into a smaller number of axes that summarize major sources of variation. This makes the neighborhood graph faster and less noisy than working directly in thousands of gene dimensions.

Here we compute 50 principal components, then inspect how much variance they explain. The neighborhood graph below will use the first 40 PCs.

In [ ]:
sc.tl.pca(adata, n_comps=50, svd_solver="arpack")

sc.pl.pca_variance_ratio(adata, log=True, n_pcs=50)

## 9. Neighborhood Graph

The neighborhood graph connects cells with similar transcriptomes in PCA space. This graph is the foundation for UMAP visualization and Leiden clustering.

Using `n_neighbors=10` emphasizes relatively local structure, while `n_pcs=40` uses enough principal components to retain broad biological signal without using every noisy dimension.

In [ ]:
sc.pp.neighbors(adata, n_neighbors=10, n_pcs=40)

## 10. UMAP Embedding

UMAP turns the high-dimensional neighborhood graph into a two-dimensional visualization. It is mainly used for exploration and communication: nearby points tend to represent transcriptionally similar cells.

UMAP coordinates should not be overinterpreted as exact distances, but they are very useful for seeing whether clusters, marker genes, or metadata categories form coherent patterns.

In [ ]:
sc.tl.umap(adata)

## 11. Leiden Clustering

Leiden clustering identifies groups of cells that are densely connected in the neighborhood graph. In PBMC data, these groups often correspond to immune cell types or subtypes.

The `resolution=0.5` setting controls cluster granularity. Higher values usually produce more clusters; lower values produce broader groups.

In [ ]:
sc.tl.leiden(adata, resolution=0.5, key_added="leiden")

sc.pl.umap(adata, color="leiden", legend_loc="on data", frameon=False, title="PBMC 3k Leiden clusters")

## 12. Marker Genes by Cluster

Marker-gene testing helps interpret clusters by identifying genes enriched in each cluster relative to the others. This is how anonymous clusters become biologically meaningful cell populations.

In the thymus project, this same idea will help identify TECs, fibroblasts, endothelial cells, and other stromal populations before comparing LOX family expression across age groups.

In [ ]:
sc.tl.rank_genes_groups(adata, groupby="leiden", method="wilcoxon")

sc.pl.rank_genes_groups(adata, n_genes=10, sharey=False)
sc.pl.rank_genes_groups_dotplot(adata, n_genes=5, groupby="leiden", standard_scale="var")

## 13. Query LOX Family Expression

PBMCs are not expected to strongly express the LOX family genes that are central to ThymusLOXScan. This section is included to test the exact query syntax we will reuse on thymic stromal data.

The code checks which requested genes are present in the AnnData object, reports missing genes, and plots expression for genes that are available. This pattern is useful because public datasets may differ in gene annotation, gene symbols, or filtering choices.

In [ ]:
lox_genes = ["LOX", "LOXL1", "LOXL2", "LOXL3", "LOXL4"]

available_lox_genes = [gene for gene in lox_genes if gene in adata.raw.var_names]
missing_lox_genes = [gene for gene in lox_genes if gene not in adata.raw.var_names]

print("Available LOX family genes:", available_lox_genes)
print("Missing LOX family genes:", missing_lox_genes)

if available_lox_genes:
    sc.pl.umap(adata, color=available_lox_genes, use_raw=True, frameon=False)
    sc.pl.violin(adata, keys=available_lox_genes, groupby="leiden", use_raw=True, rotation=90)
else:
    print("None of the requested LOX family genes are available after filtering in this PBMC dataset.")

## 14. Save Processed Tutorial Object

Saving the processed AnnData object allows later notebooks or scripts to reload the tutorial result without rerunning every preprocessing step. For the main thymus analysis, saving intermediate `.h5ad` files will make the workflow easier to audit and reproduce.

In [ ]:
adata.write("../data/processed/pbmc3k_scanpy_tutorial_processed.h5ad")